Preprocessing for meta parameter

In [1]:
import numpy as np
import os
import pydicom
import dicom_to_zarr
import helpers
import zarr
import mdreg
import time
import tqdm
from mdreg import fit_models, elastix, skimage, ants, io
import dask.array as da
import dcmri # NEUER Import für den Populations-AIF
import matplotlib.pyplot as plt # NEUER Import zum Plotten

In [2]:
DATASET_NAME = '159269_B1'
PART = '14'

In [6]:
# Define paths (please adjust if needed)
dicom_folder = f'/mnt/dev_data/{DATASET_NAME}/{PART}/DICOM'
zarr_file = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}/{PART}/{DATASET_NAME}_{PART}.zarr'
results_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}/'

In [7]:
print(dicom_folder)
print(zarr_file)
print(results_path)

/mnt/dev_data/159269_B1/14/DICOM
/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1/14/159269_B1_14.zarr
/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14/


In [8]:
# --- BENUTZEREINGABE: Bitte definieren Sie die zeitliche Auflösung Ihres Scans ---
# --- Automatische Ermittlung der Zeit-Parameter ---
time_resolution_seconds, num_time_points = helpers.get_scan_time_parameters(dicom_folder)

if time_resolution_seconds is None:
    # Stoppt das Skript, wenn die Parameter nicht gefunden werden konnten
    raise ValueError("Zeit-Parameter konnten nicht aus DICOM-Dateien ermittelt werden. Bitte überprüfen Sie die Fehlermeldungen.")

print(f"\nErmittelte Zeitauflösung: {time_resolution_seconds:.2f} s")
print(f"Ermittelte Anzahl an Zeitpunkten: {num_time_points}\n")

# 1. Erstellen Sie den Zeitvektor (`tacq`) mit np.linspace
# Wir verwenden np.linspace, um die Länge exakt auf `num_time_points` festzulegen.
# Startpunkt ist 0.
# Der Endpunkt ist (Anzahl der Punkte - 1) * Auflösung.
stop_time = (num_time_points - 1) * time_resolution_seconds
tacq = np.linspace(start=0, stop=stop_time, num=num_time_points)

# 2. Generieren Sie den Parker AIF auf diesem korrekten Zeitvektor
aif = dcmri.aif_parker(tacq)

# 3. Überprüfung der korrekten Länge
assert len(tacq) == num_time_points, "Fehler: tacq hat immer noch die falsche Länge!"
assert len(aif) == num_time_points, "Fehler: aif hat immer noch die falsche Länge!"

print("Populations-AIF (Parker) erfolgreich generiert.")
print(f"Form des AIF-Arrays: {aif.shape}")
print(f"Form des Zeit-Arrays: {tacq.shape}")

# 3. Visuelle Überprüfung des generierten AIF
plt.figure(figsize=(10, 5))
plt.plot(tacq, aif, marker='.', linestyle='-')
plt.title('Generierter Populations-AIF (Parker)')
plt.xlabel('Zeit (Sekunden)')
plt.ylabel('Kontrastmittel-Konzentration (mM)')
plt.grid(True)
plt.show()

Analysiere DICOM-Header in: /mnt/dev_data/159269_B1/14/DICOM...
Warnung: Konnte Datei /mnt/dev_data/159269_B1/14/DICOM/MR004099.dcm nicht lesen oder Tag nicht finden. Überspringe. Fehler: float() argument must be a string or a real number, not 'FileDataset'
Warnung: Konnte Datei /mnt/dev_data/159269_B1/14/DICOM/MR007670.dcm nicht lesen oder Tag nicht finden. Überspringe. Fehler: float() argument must be a string or a real number, not 'FileDataset'
Warnung: Konnte Datei /mnt/dev_data/159269_B1/14/DICOM/MR002263.dcm nicht lesen oder Tag nicht finden. Überspringe. Fehler: float() argument must be a string or a real number, not 'FileDataset'
Warnung: Konnte Datei /mnt/dev_data/159269_B1/14/DICOM/MR008524.dcm nicht lesen oder Tag nicht finden. Überspringe. Fehler: float() argument must be a string or a real number, not 'FileDataset'
Warnung: Konnte Datei /mnt/dev_data/159269_B1/14/DICOM/MR008308.dcm nicht lesen oder Tag nicht finden. Überspringe. Fehler: float() argument must be a string or

KeyboardInterrupt: 

In [9]:
aif_list = aif.tolist()
tacq_list = tacq.tolist()

NameError: name 'aif' is not defined

In [10]:
# Make sure the folder exists before calling the function
if not os.path.exists(dicom_folder):
    os.makedirs(dicom_folder)
    print(f"Folder '{dicom_folder}' was created. Please fill it with your DICOM files.")
elif not os.listdir(dicom_folder):
    print(f"The folder '{dicom_folder}' is empty. Please add DICOM files to run the script.")
else:
    # Call the conversion function
    dicom_to_zarr.convert_dicom_to_zarr(dicom_folder, zarr_file)

Lese DICOM-Dateien aus: /mnt/dev_data/159269_B1/14/DICOM
Stapele DICOM-Schichten zu einem 3D-Raum...
Extrahierte Metadaten: {'PixelSpacing': [1.9531, 1.9531], 'SliceThickness': 5.0, 'Rows': 256, 'Columns': 256, 'PatientID': 'MEDCIC_10_B1'}
Speichere Array der Größe (12500, 256, 256) in /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1/14/159269_B1_14.zarr...
Konvertierung abgeschlossen!


In [11]:
def fit_parallel(moving,
        fit_pixels = None,
        fit_coreg = None,
        fit_image = None,
        tol = 1e-6,    
        maxit = 5,
        verbose = 0,
        force_2d = False,
        path = None, 
    ):
    """
    Remove motion from a series of 2D- or 3D images.

    Parameters
    ----------
    moving : numpy.ndarray | zarr.Array
        The series of images to be corrected, with dimensions (x,y,t) or (x,y,z,t). 
    fit_pixels : dict, optional
        A dictionary defining a single-pixel signal model. The possible items 
        in the dictionary are the keywords of the function `mdreg.fit_pixels`. 
        For a slice-by-slice computation (4D array with force_2d=True), 
        *fit_pixels* can be a list of dictionaries, one for each slice. 
        The default is None.
    fit_coreg : dict, optional
        The parameters for coregistering the images. *fit_coreg* has one 
        required item 'package' with possible values 'skimage' (default), 
        'elastix' and 'ants'. The other parameters are the possible keywords 
        of the *coreg_series* function of the package specified. 
    fit_image : dict or list, optional
        A dictionary defining the function to fit the signal data, and its 
        parameter values. This argument is ignored if *fit_pixels* is already 
        provided. *fit_image* has one required key 'func' that specifies the 
        fit function to use. The other entries are the keyword arguments of 
        this fit function. 
        The fit function can be one of the functions built in to mdreg, or a 
        custom made function. A valid fit function *must* take a signal array 
        as argument, and return two variables: an array with the same shape 
        containing the fit to the model, and a second variable that contains 
        the fitted parameters. 
        For a slice-by-slice computation (4D array with force_2d=True), 
        *fit_image* can be a list of dictionaries, one for each slice. 
        If *fit_image* is not provided, a constant model is used. 
    tol : float, optional
        Stopping criterion for the iteration. The iteration stops if the 
        largest difference between new and old coregistered series in any 
        pixel at any time point is less than *tol* of the largest value. 
        The default is 1e-6.
    maxit : int, optional
        The maximum number of iterations. The default is 0.
    verbose : int, optional
        The level of feedback to provide to the user. 0: no feedback; 1: text 
        output only; 2: text output and progress bars. The default is 2.
    force_2d : bool, optional
        By default, a 3-dimensional moving array will be coregistered with a 
        3-dimensional deformation field. To perform slice-by-slice 
        2-dimensional registration instead, set *force_2d* to True. This 
        keyword is ignored when the arrays are 2-dimensional. The 
        default is False.
    path : str, optional
        Path on disk where to save the results. If no path is provided, the 
        results are not saved to disk. Defaults to None.

    Returns
    -------
    coreg : numpy.ndarray | zarr.Array
        The coregistered images with the same dimensions as *moving*.
    fit : numpy.ndarray | zarr.Array
        The fitted signal model with the same dimensions as *arr*.
    transfo : numpy.ndarray | zarr.Array | list
        The parameters of the transformation deforming the moving image to the 
        coregistered image. With skimage, this is the deformation field with 
        the same dimensions as *moving*, and one additional dimension for the 
        components of the vector field. With elastix this is an array of 
        parameter objects and with ants this is an array of files with 
        transform parameters. Note when force_2d = True these are 2-dimensional 
        arrays with one transform per slice and per time point.
    pars : numpy.ndarray | zarr.Array
        The parameters of the fitted signal model with dimensions (x,y,n) or 
        (x,y,z,n), where n is the number of free parameters of the signal 
        model.
 
    """

    print("Test")
    
    # Set defaults in fit_coreg
    if fit_coreg is None:
        fit_coreg = {'package': 'skimage'}
    if 'package' not in fit_coreg:
        fit_coreg['package'] = 'skimage'
    #if 'progress_bar' not in fit_coreg:
        #fit_coreg['progress_bar'] = verbose>1
    if 'name' not in fit_coreg:
        fit_coreg['name'] = 'coreg'

    # 2D slice-by-slice coregistration
    if moving.ndim==4:
        if force_2d:
            return  _fit_force_2d(
               moving, fit_image, fit_coreg, fit_pixels, tol, maxit, 
               verbose, path, 
            )
        
    # Set defaults for fit_image  
    if fit_image is None:
        fit_image = {'func': fit_models.fit_constant}

    # Check inputs
    if not isinstance(fit_image, dict):
        raise ValueError('The fit_image argument must be a dictionary.')

    # Set paths    
    _set_path(fit_coreg, path)
    _set_path(fit_image, path)
    _set_path(fit_pixels, path)

    # Compute
    converged = False
    it = 1
    start = time.time()

    if verbose > 0:
        print('Initializing..')
    coreg = io._copy(moving, path, 'coreg')

    while not converged: 

        startit = time.time()

        # Fit signal model
        if verbose > 0:
            print(f'Iteration {it}: fitting signal model')
        if fit_pixels is not None:
            fit, pars = fit_models.fit_pixels(coreg, **fit_pixels)
        else:
            kwargs = {i:fit_image[i] for i in fit_image if i!='func'}
            fit, pars = fit_image['func'](coreg, **kwargs)
        
        # Fit deformation
        if verbose > 0:
            print(f'Iteration {it}: fitting deformation fields')
        coreg_curr = io._copy(coreg, path, 'tmp')
        vals = _coreg_series(moving, fit, **fit_coreg)

        coreg, transfo = vals[:2]

        # Check convergence
        converged = _diff(coreg, coreg_curr) < tol
        
        if verbose > 0:
            print(f'Calculation time for iteration {it}: '
                  f'{(time.time()-startit)/60} min')  

        if it == maxit: 
            break

        it += 1 

    if verbose > 0:
        print(f'Total calculation time: {(time.time()-start)/60} min')

    io._remove(path, 'tmp')
    if len(vals) > 2: # optional return value
        defo = vals[2]
        return coreg, fit, transfo, pars, defo
    else:
        return coreg, fit, transfo, pars


def _fit_force_2d(
        moving, fit_image, fit_coreg, fit_pixels, tol, maxit, verbose, 
        path,
    ):

    # Required outputs
    coreg = io._copy(moving, path, 'coreg')
    if fit_coreg['package'] == 'skimage':
        transfo = io._defo(
            moving, 
            path, 
            force_2d=True, 
            name=fit_coreg['name']+'_defo',
        )
    else:
        transfo = np.empty(moving.shape[-2:], dtype=object)

    # Optional outputs
    defo = None
    if 'return_deformation' in fit_coreg:
        if fit_coreg['return_deformation']:
            defo = io._defo(
                moving, 
                path, 
                force_2d=True, 
                name=fit_coreg['name']+'_defo',
            )

    for k in tqdm(
            range(moving.shape[2]), 
            desc='Fitting slice', 
            disable=verbose<2,
        ):
        if verbose == 1:
            print(f'Fitting slice {k+1} / {moving.shape[2]}')

        if fit_image is None:
            fit_image_k = None
        elif isinstance(fit_image, dict):
            fit_image_k = fit_image
        else:
            fit_image_k = fit_image[k]

        if fit_pixels is None:
            fit_pixels_k = None
        elif isinstance(fit_pixels, dict):
            fit_pixels_k = fit_pixels
        else:
            fit_pixels_k = fit_pixels[k]

        vals = fit_parallel(
            moving[:,:,k,:],
            fit_pixels = fit_pixels_k,
            fit_image = fit_image_k,
            fit_coreg = fit_coreg,
            tol = tol,
            maxit = maxit,
            verbose = verbose,
        )
        coreg[:,:,k,:], fit_k, transfo_k, pars_k = vals[:4]
        if k == 0:
            fit_arr, pars = io._fit_models_init(moving, path, pars_k.shape[-1])              
        if fit_coreg['package'] == 'skimage':
            transfo[:,:,k,:,:] = transfo_k
        else:
            transfo[k,:] = transfo_k
        fit_arr[:,:,k,:] = fit_k
        pars[:,:,k,:] = pars_k
        if defo is not None:
            defo[:,:,k,:,:] = vals[4]
    if defo is None:
        return coreg, fit_arr, transfo, pars
    else:
        return coreg, fit_arr, transfo, pars, defo



def _set_path(dct, path):
    if dct is None:
        return
    if path is None:
        return
    if 'path' in dct:
        if path != dct['path']:
            raise ValueError("Two different paths are provided.")
    else:
        dct['path'] = path
        

def _diff(coreg, coreg_curr):
    if isinstance(coreg, np.ndarray):
        corr = np.max(np.abs(coreg-coreg_curr))/np.max(np.abs(coreg_curr))
    else:
        coreg = da.from_zarr(coreg) 
        coreg_curr = da.from_zarr(coreg_curr)    
        corr = da.max(da.abs(coreg-coreg_curr))/da.max(da.abs(coreg_curr))
        corr.compute()
    return corr


def _coreg_series(moving, fit, package='skimage', **fit_coreg):

    if package == 'elastix':
        print('elastix')
        fit_coreg = _set_mdreg_elastix_defaults(fit_coreg)
        return elastix.coreg_series(moving, fit, **fit_coreg)
    
    elif package == 'skimage':
        print("parallelization")
        return skimage.coreg_series(moving, fit, **fit_coreg, parallel=True, progress_bar=False)
    
    elif package == 'ants':
        print('ants')
        return ants.coreg_series(moving, fit, **fit_coreg)
    
    else:
        raise NotImplementedError(
            'This coregistration package is not implemented')
    

def _set_mdreg_elastix_defaults(params):

    if "WriteResultImage" not in params:
        params["WriteResultImage"] = "false"
    if "WriteDeformationField" not in params:
        params["WriteDeformationField"] = "false"
    if "ResultImagePixelType" not in params:
        params["ResultImagePixelType"] = "float"

    # # Removing this for v0.4.2 as results appear to be worse
    # if 'Metric' not in params:
    #     params["Metric"] = "AdvancedMeanSquares"

    # # Settings pre v0.4.0 - unclear why - removed for now
    # if "FinalGridSpacingInPhysicalUnits" not in params:
    #     params["FinalGridSpacingInPhysicalUnits"] = "50.0"
    # if "AutomaticParameterEstimation" not in params:
    #     params["AutomaticParameterEstimation"] = "true"
    # if "ASGDParameterEstimationMethod" not in params:
    #     params["ASGDParameterEstimationMethod"] = "Original"
    # if "MaximumStepLength" not in params:
    #     params["MaximumStepLength"] = "1.0"
    # if "CheckNumberOfSamples" not in params:
    #     params["CheckNumberOfSamples"] = "true"
    # if "RandomCoordinate" not in params:
    #     params["ImageSampler"] = "RandomCoordinate"

    return params

In [12]:
data = zarr.open(zarr_file)

In [13]:
temp_zarr = 'input_zarr_for_mdreg_parallel_01.zarr'

In [14]:
# --- 1. Setup: Erstelle ein Dummy-Array mit Ihrer Start-Form ---
# Dies simuliert Ihr geladenes Zarr-Array.
input_array = da.from_zarr(data)

# --- 2. Definition der Zieldimensionen ---
H, W = 256, 256 # Höhe und Breite bleiben gleich
D = 50 # 50 Bilder pro Zeitaufnahme bilden die neue Tiefen-Dimension (Z-Richtung)
T = 250 # 250 zeitlich versetzte Aufnahmen

# Überprüfung: 50 * 250 muss 12500 ergeben.
assert D * T == input_array.shape[0], "Die Dimensionen passen nicht zusammen!"

# --- 3. Umformung und Neuordnung der Achsen ---

# Schritt A: Umformen (Reshape)
# Wir interpretieren die 12500 Bilder als 250 Zeitpunkte (T) mit je 50 Tiefenschichten (D).
# Das Ergebnis hat die Form (T, D, H, W).
reshaped_array = input_array.reshape(T, D, H, W)
print(f"Form nach dem Reshape: {reshaped_array.shape}  (entspricht T, D, H, W)")

# Schritt B: Achsen neuordnen (Transpose)
# Wir bringen die Achsen in die gewünschte Reihenfolge (H, W, D, T).
# Alte Achsen: 0=T, 1=D, 2=H, 3=W
# Neue Reihenfolge: 2, 3, 1, 0
final_array = reshaped_array.transpose(2, 3, 1, 0)
print(f"Form des finalen Arrays: {final_array.shape} (entspricht H, W, D, T)")

# --- 4. Überprüfung ---
# Das finale Array hat nun die gewünschte Form (256, 256, 50, 250).
# Sie können es jetzt z.B. als neues Zarr-Array speichern.
final_array.to_zarr(temp_zarr, overwrite=True)
correct_shape_array = zarr.open(temp_zarr)

Form nach dem Reshape: (250, 50, 256, 256)  (entspricht T, D, H, W)
Form des finalen Arrays: (256, 256, 50, 250) (entspricht H, W, D, T)


In [15]:
correct_shape_array

<Array file://input_zarr_for_mdreg_parallel_01.zarr shape=(256, 256, 50, 250) dtype=int16>

In [16]:
temp = correct_shape_array[:,:,25,:]
temp = np.transpose(temp, [2,0,1])

In [17]:
helpers.explore_3D_array(temp)

interactive(children=(IntSlider(value=124, description='SLICE', max=249), Output()), _dom_classes=('widget-int…

In [28]:
# Path for output
results_path = results_path
# Die Variable `aif` kommt jetzt aus dem neuen Block oben

# HINWEIS: Da Sie nun einen sauberen, theoretischen AIF verwenden, ist die
# automatische Baseline-Ermittlung nicht mehr notwendig. Wir können die Baseline
# direkt aus der AIF-Kurve ablesen (der Punkt vor dem Anstieg).
# Für den Parker AIF ist der Anstieg sehr früh. Ein konservativer Wert wie 1 oder 2 ist sicher.
baseline_val = 2

coreg, fit, transfo, pars = fit_parallel(
    correct_shape_array,
    fit_image={
        'func': mdreg.fit_2cm_lin,
        'time': tacq_list,
        'aif': aif_list,
        'baseline': baseline_val, # Fester Baseline-Wert
        # HINWEIS: Die physikalischen Parameter fehlen hier immer noch,
        # was die quantitative Genauigkeit beeinflusst.
        # 'r1': 4.5,
        # 'signal_pars': {'TR': ..., 'TE': ..., 'FA': ..., 'hct': ...},
    },
    fit_coreg={'package':'skimage',},
    maxit=3,
    path=results_path,
    verbose=1,
)

Test
Initializing..
Iteration 1: fitting signal model


Fitting 2cm: 100%|██████████| 50/50 [06:54<00:00,  8.28s/it]


Iteration 1: fitting deformation fields
parallelization
Calculation time for iteration 1: 109.78492693901062 min
Iteration 2: fitting signal model


Fitting 2cm: 100%|██████████| 50/50 [06:56<00:00,  8.34s/it]


Iteration 2: fitting deformation fields
parallelization
Calculation time for iteration 2: 108.91112197637558 min
Iteration 3: fitting signal model


Fitting 2cm: 100%|██████████| 50/50 [07:02<00:00,  8.44s/it]


Iteration 3: fitting deformation fields
parallelization
Calculation time for iteration 3: 109.07593684593836 min
Total calculation time: 327.93489532470704 min


In [18]:
coreg = zarr.open('/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14/coreg.zarr')

In [19]:
coreg

<Array file:///mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14/coreg.zarr shape=(256, 256, 50, 250) dtype=int16>

In [20]:
correct_shape_array

<Array file://input_zarr_for_mdreg_parallel_01.zarr shape=(256, 256, 50, 250) dtype=int16>

In [21]:
coreg_vis = coreg [:,:,24,:]
coreg_vis = np.transpose(coreg_vis, [2,0,1])

In [22]:
helpers.explore_3D_array(coreg_vis)

interactive(children=(IntSlider(value=124, description='SLICE', max=249), Output()), _dom_classes=('widget-int…

In [24]:
fit_vis = fit [:,:,24,:]
fit_vis = np.transpose(fit_vis, [2,0,1])

NameError: name 'fit' is not defined

In [28]:
correct_shape = correct_shape_array [:,:,24,:]
correct_shape = np.transpose(correct_shape, [2,0,1])

In [29]:
helpers.explore_3D_array_comparison_and_diff(coreg_vis, correct_shape)

interactive(children=(IntSlider(value=125, description='Slice:', max=249), Output()), _dom_classes=('widget-in…